<a href="https://colab.research.google.com/github/renu-kg/PRODIGY_GA_01/blob/main/Task_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers
!pip install datasets

In [2]:
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from datasets import load_dataset

In [3]:
print("\nDownloading H.P. Lovecraft text...")
!wget https://www.gutenberg.org/files/50133/50133-0.txt -O lovecraft.txt
file_path = 'lovecraft.txt'
print(f"✅ Data saved to {file_path}")


--2025-10-17 14:56:30--  https://www.gutenberg.org/files/50133/50133-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 103448 (101K) [text/plain]
Saving to: ‘lovecraft.txt’

lovecraft.txt       100%[===================>] 101.02K   203KB/s    in 0.5s    

2025-10-17 14:56:32 (203 KB/s) - ‘lovecraft.txt’ saved [103448/103448]

✅ Data saved to lovecraft.txt


In [4]:
print("\nLoading pre-trained GPT-2 model and tokenizer...")
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("✅ Model and tokenizer loaded.")


Loading pre-trained GPT-2 model and tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✅ Model and tokenizer loaded.


In [5]:
print("\nPreparing dataset for training...")
# Special token to handle end-of-text
tokenizer.pad_token = tokenizer.eos_token


Preparing dataset for training...


In [6]:

train_dataset = TextDataset(
    tokenizer=tokenizer,
    file_path=file_path,
    block_size=128  # Process the text in chunks of 128 tokens
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False # Masked Language Modeling is not needed for GPT-2
)
print("✅ Dataset is ready.")

/usr/local/lib/python3.12/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (25782 > 1024). Running this sequence through the model will result in indexing errors


✅ Dataset is ready.


In [7]:
print("\nConfiguring training...")
training_args = TrainingArguments(
    output_dir="./gpt2-lovecraft", # Directory to save the new model
    overwrite_output_dir=True,
    num_train_epochs=1,           # One loop through the data is a good start
    per_device_train_batch_size=8,
    save_steps=500,
    save_total_limit=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print("\nStarting fine-tuning... (This will take several minutes)")
trainer.train()
print("✅ Fine-tuning complete!")



Configuring training...

Starting fine-tuning... (This will take several minutes)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kgrenu1 (kgrenu1-bapuji-institute-of-engineering-and-technology-d) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


✅ Fine-tuning complete!


In [9]:
print("\n--- Generating text from the new Lovecraftian model ---")
prompt = "In a realm of cosmic horror,"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids

# Move input_ids to the same device as the model
input_ids = input_ids.to(model.device)

# We use our fine-tuned 'model' to generate text
generated_output = model.generate(
    input_ids,
    max_length=150,
    num_return_sequences=1,
    temperature=0.8, # A little more creative
    top_k=50,
    do_sample=True # This enables sampling-based generation
)

# Decode and print the result
generated_text = tokenizer.decode(generated_output[0], skip_special_tokens=True)
print("\nGenerated Text:")
print(generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Generating text from the new Lovecraftian model ---

Generated Text:
In a realm of cosmic horror, the human race was now called an indeterminate, all-inclusive race. The human race as we have known it for all time, the race of the whole world, which has been determined and known under the name of the race of Homo sapiens, or Homo sapiens. The most ancient of all the unknown races came into existence during the Paleolithic period. We have found them to be some six hundred thousand years before us.

We have not yet found the only one of our race, Homo sapiens, but there are many others, many more that remain unknown. From all this it appears to be no more than an isolated, and very unknown, race, the Homo. The mystery
